In [1]:
import os
os.sys.path.append('/data/scratch/globc/bonassies/workspace/swot_for_flood')

# Download and rasterize notebook

This notebook explain how to use the swot_for_flood library to download and rasterize the SWOT data for a given region and time period. 

## Define a project

This library is designed to work with a project. A project is a directory that contains the configuration file `config.cfg` and the data. The configuration file contains the parameters of the project.

In this notebook, we will create a project named "example_download_rasterize" in the directory "examples". The project will contain the SWOT data for the region of interest and time period defined in the configuration file.

In [2]:
import configparser

config = configparser.ConfigParser()
config.read('./example_download_and_rasterize/config.cfg')

print(type(config),dict(config['CONFIG']))

<class 'configparser.ConfigParser'> {'project': 'example_download_and_rasterize', 'workspace': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples', 'data_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize/data', 'aoi': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize/aux_data/EMSR692_aois_V2.gpkg', 'crs': 'EPSG:32735', 'aoi_crs': 'EPSG:4326', 'first_time': '2023-09-15', 'last_time': '2023-09-16', 'nodes': '8', 'download_type': 'PIXC', 'passes': '[402, 417]', 'tile_names_selection': '[[417_224L], [402_085L]]', 'gdal_num_threads': '10', 'gdal_cachemax': '160000', 'do_download': 'False', 'do_make_gpkg': 'False', 'do_make_tiff': 'False'}


The config can also be a dictionary

In [ ]:
from pathlib import Path

param_dict = {
    'project': 'Greece_EMSR692',
    'workspace': Path("/data/scratch/globc/bonassies/workspace"),
    'data_path': Path("/data/scratch/globc/bonassies/data"),
    'CRS': 'EPSG:32634',
    'first_time': "2023-09-05",
    'last_time': "2023-09-20",
    'aoi': Path("/data/scratch/globc/bonassies/workspace").joinpath('Greece_EMSR692',"aux_data","EMSR692_aois_V2.gpkg"),
    'aoi_crs': 'EPSG:4326',
    'passes': [402, 417],
    'nodes': 8,
    'do_download': False,
    'download_type': 'PIXC',
    'GDAL_NUM_THREADS': 10,
    'GDAL_CACHEMAX': 160000,
    'do_make_gpkg': False,
    'do_make_tiff': False    
}

Once the config file loaded, we can use it to create a project object. The project object will contain the parameters of the project and the data.

In [3]:
from core.swot_project import SWOT_PROJECT

swot_project = SWOT_PROJECT(config)
print(swot_project)

No automatic download, please use the Downloader object to download the data
No pixc files found, please check the SWOT_PATH of download the data
Class SWOT_PROJECT():
	project: example_download_and_rasterize
	workspace: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples
	data_path: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize/data
	CRS: EPSG:32735
	first_time: 2023-09-15
	last_time: 2023-09-16
	variables: ['sig0', 'coherent_power', 'incidence', 'gamma_tot', 'gamma_SNR', 'gamma_est', 'power_plus_y', 'power_minus_y', 'interf_real', 'interf_imag', 'height', 'classification', 'bright_land_flag']
	tile_names_selection: [['417_224L'], ['402_085L']]
	PROJECT_PATH: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize
	SWOT_PATH: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize/data/SWOT
	AUX_PATH: /data/scratch/globc/bonassies/workspace/swot_fo

On the first time, you should get an error because the data is not downloaded yet. The project object will download the data and store it in the project directory only if do_download is set to True in the configuration file.

You can manually download the data by calling the download method of the project object.

First we need to search for the data we want to download. We can use the search method of the project object to search for the data. The search method will return a list of the data that match the search criteria.

In [4]:
swot_project.Downloader.search_PIXC()

Found 7 granules
Found 7 granules within [402, 417] passes


Then we can download the data by calling the download method of the project object. The download method will download the data and store it in the project directory.

In [ ]:
swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.download_granules() # if single thread download

Finally we can rasterize the data by calling the rasterize method of the project object. The rasterize method will rasterize the data and store it in the project directory.

First we create gpgk file that combine the netcdf files and then we rasterize the data.

In [ ]:
swot_project.Rasterizer.find_pixc()

swot_project.Rasterizer.pixc_to_gpkg()

swot_project.Rasterizer.gpkg_to_tiff()

If needed, we can put extra rasters to the same resolution and extent as the SWOT data. This is useful to compare the SWOT data with other data.

It uses gdal to rasterize the data. gdal is used as command line using os.system.

In [ ]:
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/example_download_and_rasterize/aux_data/FABDEM_fusion_cut_v2_32634.tif")
raster_crs = 'EPSG:32634'
interp = 'bilinear'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)

You can check other notebooks for more information about the library.